In [0]:
import logging
from pyspark.sql.types import StringType, DateType
from pyspark.sql import functions as F
from delta.tables import DeltaTable

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("main")

In [0]:
%run ./../../common/utilities

In [0]:
dbutils.widgets.text("catalog", "abcgroup", "Catalog")
dbutils.widgets.text("table", "crm_prd_info", "Table")

In [0]:
catalog = dbutils.widgets.get("catalog")
table = dbutils.widgets.get("table")

In [0]:
df = (
    spark.table(f"{catalog}.{bronze_schema}.{table}")
)
display(df.limit(5))


In [0]:
df = df.select(
    [
        F.trim(F.col(field.name)).alias(field.name)
        if isinstance(field.dataType, StringType)
        else F.col(field.name)
        for field in df.schema.fields
    ]
)

In [0]:
df = (
    df
    .withColumn(
        "prd_start_dt",
        F.col("prd_start_dt").cast(DateType())
    )
    .withColumn(
        "prd_end_dt",
        F.when(F.col("prd_end_dt").isNotNull(), F.col("prd_end_dt").cast(DateType())
        ).otherwise(F.lit("9999-12-31"))
    )
)
display(df.limit(5))

In [0]:
df = df.withColumn(
    "cat_id",
    F.when(
        F.size(F.split("prd_key", "-")) >= 2,
        F.concat_ws(
            "_",
            F.split("prd_key", "-").getItem(0),
            F.split("prd_key", "-").getItem(1)
        )
    ).otherwise(None)
)

In [0]:
df = (
    df
    .withColumn(
        "prd_key",
        F.when(
            F.length("prd_key") >= 7,
            F.substring("prd_key", 7, F.length("prd_key"))
        ).otherwise(F.col("prd_key"))
    )
)

In [0]:
df = (
    df
    .withColumn(
        "prd_cost",
        F.coalesce(F.col("prd_cost").cast("int"), F.lit(0))
    )
)
display(df.limit(5))

In [0]:
df = (
    df
    .withColumn(
        "prd_line",
        F.when(F.upper(F.col("prd_line")) == "M", "Mountain")
        .when(F.upper(F.col("prd_line")) == "R", "Road")
        .when(F.upper(F.col("prd_line")) == "S", "Other Sales")
        .when(F.upper(F.col("prd_line")) == "T", "Touring")
        .otherwise("n/a")
    )
)

In [0]:
COL_MAP = {
    "prd_id": "product_id",
    "cat_id": "category_id",
    "prd_key": "product_number",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "start_date",
    "prd_end_dt": "end_date"
}
for old_name, new_name in COL_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)
display(df.limit(5))

In [0]:
df_flagged = (
    df.withColumn(
        "is_valid",
        F.when(
            F.col("product_id").isNotNull() &
            F.col("product_name").isNotNull() &
            F.col("product_cost").isNotNull() &
            F.col("product_line").isNotNull(),
            1
        ).otherwise(0)
    )
)

In [0]:
valid_df = (
    df_flagged
    .filter(F.col("is_valid") == 1)
    .drop("is_valid")
)

invalid_df = (
    df_flagged
    .filter(F.col("is_valid") == 0)
    .drop("is_valid")
)

In [0]:
invalid_df = (
    invalid_df
    .withColumn("error_reason", F.lit("NULL in critical columns"))
    .withColumn("ingestion_ts", F.current_timestamp())
)

In [0]:
invalid_count = invalid_df.count()
if invalid_count > 0:
    logger.warning(f"{invalid_count} invalid records moved to quarantine")

    (
        invalid_df
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(f"{catalog}.{silver_schema}.{table}_quarantine")
    )

In [0]:
display(valid_df.limit(5))

In [0]:
target_table = f"{catalog}.{silver_schema}.crm_products"
valid_df.write.mode("overwrite").format("delta").saveAsTable(target_table)

logger.info("Saved table into %s in Delta format.", target_table)

In [0]:
display(valid_df.limit(5))

In [0]:
display(spark.sql(f"""
    SELECT *
    FROM {catalog}.{silver_schema}.crm_products
    LIMIT 5
"""))